In [10]:
import json
import os

import geopandas as gpd
import momepy as mp
import networkx as nx

from paris_orderbike.growth import Orderbike

In [11]:
import pandas as pd

In [20]:
df = pd.read_json("../data/processed/2021/bs_400_directness/metrics_growth.json")

In [21]:
df

,xx,directness,coverage,num_cc,length_lcc,gini
0,124300.553691,0.365075,7.321742e+07,17,101464.182090,0.248682
1,124658.243420,0.401481,7.321742e+07,16,108009.438881,0.248959
2,125173.995071,0.431956,7.322021e+07,15,111582.278480,0.245376
3,125518.008061,0.458838,7.322996e+07,14,115578.427315,0.243272
4,125958.786141,0.483645,7.322996e+07,13,118567.008725,0.242974
...,...,...,...,...,...,...
385,299113.090504,0.864350,8.940619e+07,1,299113.090504,0.114300
386,299298.037875,0.864725,8.940619e+07,1,299298.037875,0.114303
387,299592.637154,0.864771,8.940619e+07,1,299592.637154,0.114306
388,300241.756583,0.863755,8.940619e+07,1,300241.756583,0.114199


In [6]:
FOLDEROOT = "../data/processed/"
END_FOLDERS = [
    "Nothing",
    "2021",
    "2026",
]
NUM_RAND_TRIAL = 1000
BUFFER_SIZE = 400

In [7]:
def init_gdf(gdf_edges, end_folder):
    gdf = gdf_edges.copy()
    if end_folder == "2021":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x == "2021-01-01" else 0)
    elif end_folder == "2026":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x != "No" else 0)
    elif end_folder == "Nothing":
        G = mp.gdf_to_nx(gdf, integer_labels=False, preserve_index=True)
        closeness = nx.closeness_centrality(G, distance="length")
        edge_closeness = {
            edge: (closeness[edge[0]] + closeness[edge[1]]) / 2
            for edge in G.edges
            if (
                (G.edges[edge]["highway"] == "primary")
                & (G.edges[edge]["level"] == "primary")
                & (G.edges[edge]["built_in"] == "2021-01-01")
            )
        }
        choice = max(edge_closeness, key=edge_closeness.get)
        gdf["built"] = gdf.apply(
            lambda df: (
                1
                if (
                    ((df["from"] == str(choice[0])) & (df["to"] == str(choice[1])))
                    or ((df["to"] == str(choice[0])) & (df["from"] == str(choice[1])))
                )
                else 0
            ),
            axis=1,
        )
    return gdf


def run_and_save_orderbike(G, met, buffer_size, foldername, recompute):
    if os.path.exists(foldername) and not recompute:
        pass
    else:
        if not os.path.exists(foldername):
            os.makedirs(foldername)
        odb = Orderbike(
            G,
            preset=met,
            buff_size=buffer_size,
            built=True,
            keep_connected=False,
        )
        odb.grow()
        order_growth = odb.get_growth_order()
        met_dict = odb.get_metrics_dict()
        with open(foldername + "order_growth.json", "w") as f:
            json.dump(order_growth, f)
        with open(foldername + "metrics_growth.json", "w") as f:
            json.dump(met_dict, f)

In [8]:
gdf_raw = gpd.read_file(FOLDEROOT + "bikenet_edges.gpkg")

In [9]:
for end_folder in END_FOLDERS:
    folder_save = FOLDEROOT + end_folder + "/"
    if not os.path.exists(folder_save):
        os.makedirs(folder_save)
    gdf_edges = init_gdf(gdf_raw, end_folder)
    G = mp.gdf_to_nx(gdf_edges, integer_labels=False, preserve_index=True)
    for i in range(NUM_RAND_TRIAL):
        foldername = (
            folder_save
            + f"bs_{BUFFER_SIZE}_random_disconnected/random_disconnected_{i:03}/"
        )
        run_and_save_orderbike(
            G,
            "random",
            BUFFER_SIZE,
            foldername,
            False,
        )